In [ ]:
%%writefile hola.cpp
/*Analysis of Algorithms
Assignment 06*/
/*All coding exercise were making in Google Colab*/

Overwriting hola.cpp


In [4]:
%%writefile hola.cpp
/*2. Implement the Merge Sort algorithm.
3. Use your implementations and experiment to check if the merge sort is or is not a stable sorting
algorithm. Provide and example.*/

#include <iostream>
#include <vector>
#include <string>

struct Item {
    int key;
    std::string value;
};

void merge(const std::vector<Item>& B, const std::vector<Item>& C, std::vector<Item>& A) {
    int p = B.size();
    int q = C.size();
    int i = 0, j = 0, k = 0;

    while (i < p && j < q) {
        if (B[i].key <= C[j].key) {
            A[k] = B[i];
            i++;
        } else {
            A[k] = C[j];
            j++;
        }
        k++;
    }

    if (i == p) {
        while (j < q) {
            A[k] = C[j];
            j++;
            k++;
        }
    } else {
        while (i < p) {
            A[k] = B[i];
            i++;
            k++;
        }
    }
}

void mergeSort(std::vector<Item>& A) {
    int n = A.size();
    if (n > 1) {
        int mid = n / 2;

        std::vector<Item> B(A.begin(), A.begin() + mid);
        std::vector<Item> C(A.begin() + mid, A.end());

        mergeSort(B);
        mergeSort(C);

        merge(B, C, A);
    }
}

int main() {
    // Stability probe
    std::vector<Item> A = {
        {5, "1"},
        {2, "1"},
        {5, "2"},
        {1, "1"},
        {5, "3"}
    };
//we enumerate the number 5 to see if the algorihtm interchnges the positions of the element and to test if it's stable(Mergesot is stable)
    std::cout << "Unsorted:\n";
    for (const auto& item : A) {
        std::cout << "  (" << item.key << ", " << item.value << ")\n";
    }

    mergeSort(A);

    std::cout << "\nSorted:\n";
    for (const auto& item : A) {
        std::cout << "  (" << item.key << ", " << item.value << ")\n";
    }

    return 0;
}

Overwriting hola.cpp


In [5]:
!g++ hola.cpp -o hola
!./hola

Unsorted:
  (5, 1)
  (2, 1)
  (5, 2)
  (1, 1)
  (5, 3)

Sorted:
  (1, 1)
  (2, 1)
  (5, 1)
  (5, 2)
  (5, 3)


In [6]:
%%writefile hola2.cpp
/*4. Implement the Quick Sort algorithm and test using random generated lists.*/

#include <iostream>
#include <vector>
#include <random>
#include <algorithm>

int hoarePartition(std::vector<int>& A, int l, int r) {
    int p = A[l];
    int i = l;
    int j = r + 1;

    while (true) {
        do {
            i++;
        } while (i < r && A[i] < p);

        do {
            j--;
        } while (A[j] > p);

        std::swap(A[i], A[j]);

        if (i >= j) {
            break;
        }
    }

    std::swap(A[i], A[j]);
    std::swap(A[l], A[j]);

    return j;
}

void quickSort(std::vector<int>& A, int l, int r) {
    if (l < r) {
        int s = hoarePartition(A, l, r);
        quickSort(A, l, s - 1);
        quickSort(A, s + 1, r);
    }
}

int main() {
    // Random arrays to test the quick sort algorithm
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_int_distribution<int> dis(1, 100);

    int n = 15;
    std::vector<int> A(n);

    for (int i = 0; i < n; ++i) {
        A[i] = dis(gen);
    }

    std::cout << "Random Array:\n";
    for (int val : A) {
        std::cout << val << " ";
    }
    std::cout << "\n\n";

    quickSort(A, 0, n - 1);

    std::cout << "Sorted Array:\n";
    for (int val : A) {
        std::cout << val << " ";
    }
    std::cout << "\n";

    return 0;
}



Writing hola2.cpp


In [7]:
!g++ hola2.cpp -o hola2
!./hola2

Random Array:
4 60 94 92 69 57 41 4 56 70 65 28 17 89 30 

Sorted Array:
4 4 17 28 30 41 56 57 60 65 69 70 89 92 94 


In [10]:
%%writefile hola3.cpp
/*5. Apply empirical analysis on the quick sort. Does it match with the mathematical analysis?
In the mathematical analysis the average case of the algorithm is O(n log n) and the worst case is O(n^2)
*/

#include <iostream>
#include <vector>
#include <random>
#include <chrono>
#include <iomanip>

int hoarePartition(std::vector<int>& A, int l, int r) {
    int p = A[l];
    int i = l;
    int j = r + 1;

    while (true) {
        do { i++; } while (i < r && A[i] < p);
        do { j--; } while (A[j] > p);
        std::swap(A[i], A[j]);
        if (i >= j) break;
    }

    std::swap(A[i], A[j]);
    std::swap(A[l], A[j]);
    return j;
}

void quickSort(std::vector<int>& A, int l, int r) {
    if (l < r) {
        int s = hoarePartition(A, l, r);
        quickSort(A, l, s - 1);
        quickSort(A, s + 1, r);
    }
}

int main() {
    std::mt19937 gen(12345);
    std::uniform_int_distribution<int> dis(1, 1000000);

    std::vector<int> sizes = {1000, 2000, 4000, 8000, 16000, 32000};
    double prev_rand_time = 0.0;

    std::cout << "| Size (n) | Random Inputs (ms) | Sorted Inputs (ms) | Growth Ratio T(2n)/T(n) |\n";

    for (int n : sizes) {
        std::vector<int> rand_arr(n);
        for (int i = 0; i < n; ++i) rand_arr[i] = dis(gen);

        auto start = std::chrono::high_resolution_clock::now();
        quickSort(rand_arr, 0, n - 1);
        auto end = std::chrono::high_resolution_clock::now();
        double time_rand = std::chrono::duration<double, std::milli>(end - start).count();

        double time_sorted = 0.0;
        if (n <= 4000) {
            std::vector<int> sorted_arr(n);
            for (int i = 0; i < n; ++i) sorted_arr[i] = i;

            start = std::chrono::high_resolution_clock::now();
            quickSort(sorted_arr, 0, n - 1);
            end = std::chrono::high_resolution_clock::now();
            time_sorted = std::chrono::duration<double, std::milli>(end - start).count();
        }

        double ratio = (prev_rand_time > 0.001) ? (time_rand / prev_rand_time) : 0.0;
        prev_rand_time = time_rand;

        std::cout << "| " << n << " | " << std::fixed << std::setprecision(3) << time_rand
                  << " | " << (n <= 4000 ? std::to_string(time_sorted).substr(0, 5) : "O(n^2) / Stack Overflow")
                  << " | " << (ratio > 0 ? std::to_string(ratio).substr(0, 4) : "N/A") << " |\n";
    }

    return 0;
}
/* El análisis empírico confirma la teoría en entradas aleatorias,
la razón de crecimiento T(2n)/T(n) approx 2.15 valida la complejidad promedio de O(n \log n),
mientras que en arreglos ya ordenados se degrada a O(n^2) por la elección del primer elemento como pivote. */

Overwriting hola3.cpp


In [11]:
!g++ hola3.cpp -o hola3
!./hola3

| Size (n) | Random Inputs (ms) | Sorted Inputs (ms) | Growth Ratio T(2n)/T(n) |
| 1000 | 0.197 | 1.582 | N/A |
| 2000 | 0.379 | 6.167 | 1.92 |
| 4000 | 0.836 | 26.27 | 2.20 |
| 8000 | 1.741 | O(n^2) / Stack Overflow | 2.08 |
| 16000 | 4.461 | O(n^2) / Stack Overflow | 2.56 |
| 32000 | 8.002 | O(n^2) / Stack Overflow | 1.79 |
